# Banking Intent Assistant on BANKING77

PCAM ZG516 Applied Generative AI, Group Major Project, Assignment 1.

This notebook walks through the full workflow in order: dataset, model, baseline (E1), prompt comparison (E2),
decoding comparison (E3), structured output (E4), adaptation by retrieval against a classical reference (Req 8),
failure analysis (E6), the final tool-assisted assistant (E7), and its one-time evaluation on a held-out subset.

Every experiment lives in a `.py` file in the repository. The notebook imports those files; it does not copy the code.
Every experiment cell calls the model on the 100-row subset and recomputes the results.
Total time: about 25 minutes on an M3 Mac, about 8 minutes on a Colab T4 GPU.
To read the results in about one minute, skip the `%run` cells; the result cells load the saved `results_*.csv` files.

Full explanation of every step: `docs/PROGRESS.md`. Failure analysis: `docs/FAILURES.md`.

## 0. Setup

On Google Colab this clones the repository into the session, installs the dependencies, and mounts Google Drive for one
folder only: `evaluation/results/` becomes a link to `MyDrive/banking-agent-results/<model name>/`. Every result CSV is
written to Drive directly and stays there after the session ends; the code and data are not copied to Drive.
Locally (after `uv sync`) it only changes to the project root.


In [1]:
import os, sys

# One variable selects the model for every cell below. Default: Qwen/Qwen2.5-1.5B-Instruct.
# To compare another Hugging Face chat model, set it here before any other cell runs.
# The model must have a chat template that accepts `tools` (main.py passes tool schemas to it).
# os.environ["MODEL"] = "Qwen/Qwen2.5-3B-Instruct"

if "google.colab" in sys.modules:
    !git clone -q https://github.com/2025paml556-bhoomika/banking-agent.git
    %cd banking-agent
    %pip install -q transformers torch jsonschema pandas scikit-learn
    # Keep the result files in Drive, one folder per model. Nothing else is copied.
    from google.colab import drive
    drive.mount("/content/drive")
    results = "/content/drive/MyDrive/banking-agent-results/" + os.environ.get("MODEL", "Qwen/Qwen2.5-1.5B-Instruct").split("/")[-1]
    os.makedirs(results, exist_ok=True)
    !rm -rf evaluation/results && ln -s {results} evaluation/results
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # run from the project root so imports and evaluation/ paths work

import pandas as pd
pd.set_option("display.max_colwidth", 80)


## 1. Dataset (Req 2)

BANKING77: 13,083 online-banking customer messages, each labelled with one of 77 fine-grained intents.
Two fields: `text` and `category`. The dataset has a fixed train/test split.

We use the **train split** (10,003 rows) only as a source of labelled examples for few-shot prompts and retrieval.
Two fixed 100-row samples of the test split are used for evaluation:

- `eval_subset.csv` (seed 0): the development set. Every experiment below runs on it, and it chose the final system.
- `heldout_subset.csv` (seed 1, no overlap): the held-out set. Used once, by the final system only, in section 9.

The next cell rebuilds both files from `data/test.csv`. The seeds are fixed, so the rows are the same as the committed files.


In [2]:
%run -m evaluation.make_eval_subset

test split: 3080 rows, 77 intents
development subset: 100 rows, 53 intents
held-out subset: 100 rows, 57 intents, overlap with development: 0


In [3]:
train, test = pd.read_csv("data/train.csv"), pd.read_csv("data/test.csv")
subset = pd.read_csv("evaluation/eval_subset.csv")

print(f"train: {len(train)} rows, test: {len(test)} rows, intents: {train.category.nunique()}")
print(f"eval subset: {len(subset)} rows covering {subset.category.nunique()} intents")
print(f"message length (words): mean {train.text.str.split().str.len().mean():.1f}, max {train.text.str.split().str.len().max()}")
subset.head(8)

train: 10003 rows, test: 3080 rows, intents: 77
eval subset: 100 rows covering 53 intents
message length (words): mean 11.9, max 79


,text,category
0,How do I track my card?,card_arrival
1,How long should my new card take to arrive?,card_arrival
2,What foreign exchange rate will you use?,exchange_rate
3,when does the rate get determined,exchange_rate
4,Why am I unable to take money from my card but the statement says the transa...,pending_cash_withdrawal
5,I need my card quickly,card_delivery_estimate
6,When can I expect to receive my new card?,card_delivery_estimate
7,I can't get my card to work.,card_not_working


In [4]:
# Intent distribution in the train split (top and bottom 5) and one quirk: a single label with a capital letter.
counts = train.category.value_counts()
display(pd.concat([counts.head(5), counts.tail(5)]).to_frame("train rows"))
print("labels with a capital letter:", [c for c in counts.index if c != c.lower()])

,train rows
category,
card_payment_fee_charged,187
direct_debit_payment_not_recognised,182
balance_not_updated_after_cheque_or_cash_deposit,181
wrong_amount_of_cash_received,180
cash_withdrawal_charge,177
lost_or_stolen_card,82
card_swallowed,61
card_acceptance,59
virtual_card_not_working,41


labels with a capital letter: ['Refund_not_showing_up']


## 2. Model (Req 4, Req 5)

`llm.py` loads a pretrained Hugging Face causal language model, `Qwen/Qwen2.5-1.5B-Instruct`, and exposes one
function, `generate(messages, max_new_tokens, tools, **decoding)`. Every experiment and the final assistant call
this function. Nothing else touches the model.

The cell below shows the four stages of autoregressive generation on a project example:
tokenization, embeddings, the language-modelling head (next-token logits), and the generation loop.

In [5]:
import torch
from src.llm import tokenizer, model, generate, DEVICE, MODEL

message = "How do I track my card?"
ids = tokenizer(message)["input_ids"]
print("1. tokenization:", [tokenizer.decode([i]) for i in ids], "->", ids)

emb = model.get_input_embeddings()(torch.tensor([ids]).to(DEVICE))
print("2. embeddings:  ", tuple(emb.shape), "(1 sequence, tokens, hidden size)")

with torch.no_grad():
    logits = model(torch.tensor([ids]).to(DEVICE)).logits[0, -1]
top = torch.topk(logits.float().softmax(-1), 5)
print("3. LM head, top-5 next tokens after the message:", [(tokenizer.decode([i]), round(p.item(), 3)) for p, i in zip(top.values, top.indices)])

print("4. autoregressive generation (greedy, chat format):")
print("  ", generate([{"role": "user", "content": message}], max_new_tokens=40))

/Users/anujkumar/wilp_AIML/banking_agent/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


00:08:39 llm: loading Qwen/Qwen2.5-1.5B-Instruct on mps


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 8655.60it/s]

00:08:55 llm: model ready, dtype torch.bfloat16, 1.5B parameters


...

2. embeddings:   (1, 7, 1536) (1 sequence, tokens, hidden size)


00:09:00 llm: LLM call: 36 prompt tokens, decoding={'do_sample': False}


3. LM head, top-5 next tokens after the message: [(' I', 0.097), (' To', 0.097), (' How', 0.075), (' You', 0.059), (' A', 0.031)]
4. autoregressive generation (greedy, chat format):


00:09:02 llm: LLM output (40 new tokens): "I'm sorry, but as an AI language model, I don't have access to your personal information or financial details. It's important to keep your sensitive data secure and not share it with anyone without"


   I'm sorry, but as an AI language model, I don't have access to your personal information or financial details. It's important to keep your sensitive data secure and not share it with anyone without


## 3. E1 Baseline (Req 5)

One simple prompt: the task, the 77 intent names, the message. No role, no examples, no output rules. Greedy decoding.
The output is free text, so `parse_intent` finds the longest intent name inside it.

The results cell prints the accuracy, the rows without a valid intent, and the rows where the model added preamble text.
The classical reference (TF-IDF + logistic regression, section 7) is the bar the LLM must beat on the same rows.


Run the experiment (about 2 minutes on an M3 Mac, less on a GPU):

In [6]:
%run -m src.baseline

00:09:02 baseline: loaded 77 intent names from the BANKING77 train split


00:09:02 baseline: E1 baseline: 100 test messages, greedy decoding, max 32 new tokens


00:09:02 baseline: --- row 1/100: 'How do I track my card?' (expected card_arrival)


00:09:02 llm: LLM call: 424 prompt tokens, decoding={'do_sample': False}


00:09:04 llm: LLM output (19 new tokens): 'The intent for the message "How do I track my card?" is:\n\nCard_delivery_estimate'


...

00:10:47 llm: LLM call: 432 prompt tokens, decoding={'do_sample': False}


00:10:47 llm: LLM output (4 new tokens): 'activate_my_card'


00:10:47 baseline: parse_intent: found ['activate_my_card'] in output -> 'activate_my_card'


00:10:48 baseline: saved evaluation/results/results_baseline.csv with columns ['text', 'category', 'output', 'predicted', 'correct']


accuracy: 0.41
no intent found: 4

message:  How do I track my card?
output:   'The intent for the message "How do I track my card?" is:\n\nCard_delivery_estimate'
expected: card_arrival

message:  How long should my new card take to arrive?
output:   'Card_arrival'
expected: card_arrival

message:  What foreign exchange rate will you use?
output:   'The intent for the message "What foreign exchange rate will you use?" is:\n\nexchange_rate'
expected: exchange_rate


Results:

In [7]:
from src.baseline import build_prompt, parse_intent, INTENTS

print(build_prompt("How do I track my card?")[0]["content"][:200], "...\n")
r = pd.read_csv("evaluation/results/results_baseline.csv")
print(f"accuracy {r.correct.mean():.2f}, no valid intent {r.predicted.isna().sum()}, outputs with preamble {r.output.str.match(r'^\s*The').sum()}")
r[["text", "category", "output", "predicted", "correct"]].head(5)

00:10:48 baseline: loaded 77 intent names from the BANKING77 train split


Classify the customer message into one of these intents:
Refund_not_showing_up, activate_my_card, age_limit, apple_pay_or_google_pay, atm_support, automatic_top_up, balance_not_updated_after_bank_tran ...

accuracy 0.41, no valid intent 4, outputs with preamble 57


,text,category,output,predicted,correct
0,How do I track my card?,card_arrival,"The intent for the message ""How do I track my card?"" is:\n\nCard_delivery_es...",card_delivery_estimate,False
1,How long should my new card take to arrive?,card_arrival,Card_arrival,card_arrival,True
2,What foreign exchange rate will you use?,exchange_rate,"The intent for the message ""What foreign exchange rate will you use?"" is:\n\...",exchange_rate,True
3,when does the rate get determined,exchange_rate,"The intent for the message ""when does the rate get determined"" is not explic...",NaN,False
4,Why am I unable to take money from my card but the statement says the transa...,pending_cash_withdrawal,"The intent for this message is ""pending_transfer"".",pending_transfer,False


## 4. E2 Prompt comparison (Req 6, Req 11)

`prompts.build_prompt(message, style)` is the reusable, parameterised template. Five styles, each adding one thing to the baseline:
role line, constraint line ("exact intent name, nothing else, always choose one"), 5 random solved examples, and examples + constraint.

Two things to compare in the results table: accuracy, and format (`no_intent`, `preamble`). Instructions change the format;
they cannot supply knowledge the model lacks, namely where the boundary between near-synonym labels lies.


Run the experiment (about 8 minutes on an M3 Mac, less on a GPU):

In [8]:
%run -m src.prompts

00:10:48 llm: LLM call: 424 prompt tokens, decoding={'do_sample': False}


00:10:49 llm: LLM output (19 new tokens): 'The intent for the message "How do I track my card?" is:\n\nCard_delivery_estimate'


00:10:49 llm: LLM call: 427 prompt tokens, decoding={'do_sample': False}


00:10:49 llm: LLM output (4 new tokens): 'Card_arrival'


00:10:50 llm: LLM call: 425 prompt tokens, decoding={'do_sample': False}


...

00:18:03 baseline: parse_intent: found ['verify_my_identity'] in output -> 'verify_my_identity'


00:18:03 baseline: parse_intent: found ['verify_my_identity'] in output -> 'verify_my_identity'


00:18:03 baseline: parse_intent: found ['verify_my_identity'] in output -> 'verify_my_identity'


00:18:03 baseline: parse_intent: found ['get_physical_card'] in output -> 'get_physical_card'



== few_shot_constraint: accuracy 0.43
  'How do I track my card?' -> 'card_delivery_estimate' (expected card_arrival)
  'How long should my new card take to arrive?' -> 'card_arrival' (expected card_arrival)
  'What foreign exchange rate will you use?' -> 'exchange_rate' (expected exchange_rate)

             prompt  accuracy  no_intent  preamble
          zero_shot      0.41          4        57
               role      0.40          4        30
         constraint      0.40          2         0
           few_shot      0.43         12         1
few_shot_constraint      0.43          8         0


Results:

In [9]:
from src.prompts import STYLES, build_prompt as prompt_by_style

rows = []
for s in STYLES:
    r = pd.read_csv(f"evaluation/results/results_{s}.csv")
    rows.append({"prompt": s, "accuracy": r.correct.mean(), "no_intent": r.predicted.isna().sum(), "preamble": r.output.str.match(r"^\s*The").sum()})
display(pd.DataFrame(rows))

print("--- constraint prompt (tail) ---")
print(prompt_by_style("How do I track my card?", "constraint")[0]["content"][-250:])

,prompt,accuracy,no_intent,preamble
0,zero_shot,0.41,4,57
1,role,0.40,4,30
2,constraint,0.40,2,0
3,few_shot,0.43,12,1
4,few_shot_constraint,0.43,8,0


--- constraint prompt (tail) ---
sa_or_mastercard, why_verify_identity, wrong_amount_of_cash_received, wrong_exchange_rate_for_cash_withdrawal
Reply with the exact intent name from the list and nothing else. Always choose exactly one intent.

Message: How do I track my card?
Intent:


## 5. E3 Decoding comparison (Req 7)

Same prompt (`few_shot_constraint`), five decoding settings: greedy, temperature 0.7, temperature 1.5, top-k 50, top-p 0.9. Sampling runs are seeded.

A one-label answer has one correct token sequence, so randomness can only move away from the model's best guess; the
`rows flipped vs greedy` column shows how many answers each setting changed. Greedy is used for the intent step in the final
system; sampling is used only for the free-text reply.


Run the experiment (about 8 minutes on an M3 Mac, less on a GPU):

In [10]:
%run -m src.decoding

00:18:03 llm: LLM call: 582 prompt tokens, decoding={'do_sample': False}


00:18:04 llm: LLM output (4 new tokens): 'card_delivery_estimate'


00:18:04 llm: LLM call: 585 prompt tokens, decoding={'do_sample': False}


00:18:04 llm: LLM output (4 new tokens): 'card_arrival'


00:18:04 llm: LLM call: 583 prompt tokens, decoding={'do_sample': False}


...

00:26:31 baseline: parse_intent: found ['verify_my_identity'] in output -> 'verify_my_identity'


00:26:31 baseline: parse_intent: found ['verify_my_identity'] in output -> 'verify_my_identity'


00:26:31 baseline: parse_intent: found ['verify_my_identity'] in output -> 'verify_my_identity'


00:26:31 baseline: parse_intent: found ['get_physical_card'] in output -> 'get_physical_card'



== top_p_0.9: accuracy 0.38
  'How do I track my card?' -> 'card_delivery_estimate' (expected card_arrival)
  'How long should my new card take to arrive?' -> 'card_arrival' (expected card_arrival)
  'What foreign exchange rate will you use?' -> 'exchange_rate' (expected exchange_rate)

       decoding  accuracy  no_intent  mean_output_chars
         greedy      0.43          8               21.9
temperature_0.7      0.44         10               22.0
temperature_1.5      0.40          7               22.5
       top_k_50      0.44          7               22.0
      top_p_0.9      0.38         11               21.1


Results:

In [11]:
from src.decoding import SETTINGS

g = pd.read_csv("evaluation/results/results_decoding_greedy.csv")
rows = []
for name in SETTINGS:
    r = pd.read_csv(f"evaluation/results/results_decoding_{name}.csv")
    rows.append({"decoding": name, "params": SETTINGS[name], "accuracy": r.correct.mean(), "no_intent": r.predicted.isna().sum(), "rows flipped vs greedy": (r.correct != g.correct).sum()})
pd.DataFrame(rows)

,decoding,params,accuracy,no_intent,rows flipped vs greedy
0,greedy,{'do_sample': False},0.43,8,0
1,temperature_0.7,"{'do_sample': True, 'temperature': 0.7}",0.44,10,5
2,temperature_1.5,"{'do_sample': True, 'temperature': 1.5}",0.40,7,13
3,top_k_50,"{'do_sample': True, 'temperature': 1.0, 'top_k': 50}",0.44,7,7
4,top_p_0.9,"{'do_sample': True, 'temperature': 1.0, 'top_k': 0, 'top_p': 0.9}",0.38,11,11


## 6. E4 Structured output (Req 9)

The model is asked for a JSON object `{"intent": ..., "confidence": ...}`. Every output is validated with `jsonschema` against a schema
whose `intent` field is an `enum` of the 77 names. The enum turns "is this a real intent?" into a hard check.

The results cell separates "valid JSON" from "passes the schema": the gap between the two is the number of invented intent names.
It also prints the distribution of `confidence`, to check whether the model varies it or copies the example value.


Run the experiment (about 3 minutes on an M3 Mac, less on a GPU):

In [12]:
%run -m src.structured

00:26:34 llm: LLM call: 650 prompt tokens, decoding={'do_sample': False}


00:26:48 llm: LLM output (21 new tokens): '{\n  "intent": "card_delivery_estimate",\n  "confidence": 0.9\n}'


00:26:48 llm: LLM call: 653 prompt tokens, decoding={'do_sample': False}


00:26:50 llm: LLM output (21 new tokens): '{\n  "intent": "card_arrival",\n  "confidence": 0.9\n}'


00:26:50 llm: LLM call: 651 prompt tokens, decoding={'do_sample': False}


...

00:29:50 llm: LLM call: 655 prompt tokens, decoding={'do_sample': False}


00:29:52 llm: LLM output (26 new tokens): '```json\n{\n  "intent": "unable_to_verify_identity",\n  "confidence": 0.9\n}\n```'


00:29:52 llm: LLM call: 658 prompt tokens, decoding={'do_sample': False}


00:29:54 llm: LLM output (21 new tokens): '{\n  "intent": "get_physical_card",\n  "confidence": 0.9\n}'


valid JSON:        100/100
valid vs schema:   86/100
accuracy (all):    0.43
accuracy (valid):  0.50
mean confidence when right: 0.91
mean confidence when wrong: 0.91

validation errors:
error
schema: $.intent = 'get_virtual_card' is invalid         4
schema: $.intent = 'top_up_by_card' is invalid           4
schema: $.intent = 'refund_not_showing_up' is invalid    3
schema: $.intent = 'top_up_by_cheque' is invalid         2
schema: $.intent = 'exchange_currency' is invalid        1

examples:
  'How do I track my card?' -> '{\n  "intent": "card_delivery_estimate",\n  "confidence": 0.9\n}' (expected card_arrival)
  'How long should my new card take to arrive?' -> '{\n  "intent": "card_arrival",\n  "confidence": 0.9\n}' (expected card_arrival)
  'What foreign exchange rate will you use?' -> '```json\n{\n  "intent": "exchange_rate",\n  "confidence": 0.9\n}\n```' (expected exchange_rate)


Results:

In [13]:
import json
from src.structured import SCHEMA, parse_json

print(json.dumps({**SCHEMA, "properties": {**SCHEMA["properties"], "intent": {"type": "string", "enum": "<77 intent names>"}}}, indent=2))

r = pd.read_csv("evaluation/results/results_structured.csv")
valid = r.error.isna()
json_ok = (~r.error.str.startswith("not JSON", na=False)).sum()
print(f"\nvalid JSON {json_ok}/100, pass schema {valid.sum()}/100, accuracy all {r.correct.mean():.2f}, accuracy valid rows {r[valid].correct.mean():.2f}")
print("confidence values:", r.confidence.value_counts().to_dict())
print("\nrejected by schema:")
display(r[~valid][["text", "category", "output", "error"]].head(5))
print("validated samples:")
r[valid][["text", "category", "output", "correct"]].head(3)

{
  "type": "object",
  "properties": {
    "intent": {
      "type": "string",
      "enum": "<77 intent names>"
    },
    "confidence": {
      "type": "number",
      "minimum": 0,
      "maximum": 1
    }
  },
  "required": [
    "intent",
    "confidence"
  ],
  "additionalProperties": false
}

valid JSON 100/100, pass schema 86/100, accuracy all 0.43, accuracy valid rows 0.50
confidence values: {0.9: 84, 0.95: 16}

rejected by schema:


,text,category,output,error
10,How do I exchange currency?,exchange_via_app,"```json\n{\n ""intent"": ""exchange_currency"",\n ""confidence"": 0.9\n}\n```",schema: $.intent = 'exchange_currency' is invalid
22,help me obtain a virtual card,getting_virtual_card,"{\n ""intent"": ""get_virtual_card"",\n ""confidence"": 0.95\n}",schema: $.intent = 'get_virtual_card' is invalid
23,Where is the virtual card located?,getting_virtual_card,"{\n ""intent"": ""get_virtual_card"",\n ""confidence"": 0.9\n}",schema: $.intent = 'get_virtual_card' is invalid
24,Can I order a virtual card?,getting_virtual_card,"```json\n{\n ""intent"": ""get_virtual_card"",\n ""confidence"": 0.95\n}\n```",schema: $.intent = 'get_virtual_card' is invalid
25,Where are the virtual cards located?,getting_virtual_card,"{\n ""intent"": ""get_virtual_card"",\n ""confidence"": 0.9\n}",schema: $.intent = 'get_virtual_card' is invalid


validated samples:


,text,category,output,correct
0,How do I track my card?,card_arrival,"{\n ""intent"": ""card_delivery_estimate"",\n ""confidence"": 0.9\n}",False
1,How long should my new card take to arrive?,card_arrival,"{\n ""intent"": ""card_arrival"",\n ""confidence"": 0.9\n}",True
2,What foreign exchange rate will you use?,exchange_rate,"```json\n{\n ""intent"": ""exchange_rate"",\n ""confidence"": 0.9\n}\n```",True


## 7. Adaptation decision: retrieval few-shot (Req 8)

Prompting (E2) and decoding (E3) do not add knowledge. The failures are near-synonym confusions, which is missing knowledge, not missing instructions.
Three options: more prompting, retrieval (RAG), or fine-tuning (PEFT). We test retrieval: for each message, the 5 most similar
train messages (TF-IDF cosine) become the few-shot examples.

The results table compares four systems on the same 100 rows: the LLM with 5 random examples (E2), a 5-NN vote on the retrieved
examples without the LLM, the classical TF-IDF + logistic-regression classifier, and the LLM with the 5 retrieved examples.
The decision rule: if the retrieval system passes the classical reference, fine-tuning is not needed and the adaptation is RAG.


Run the classical reference (TF-IDF + logistic regression, trained on the full train split, seconds, no LLM):


In [14]:
%run -m evaluation.reference_classifier

trained on 10003 train rows, scored on 100 development rows
accuracy: 0.79


Run the experiment (about 2 minutes on an M3 Mac, less on a GPU):

In [15]:
%run -m src.retrieval

00:30:03 llm: LLM call: 530 prompt tokens, decoding={'do_sample': False}


00:30:14 llm: LLM output (6 new tokens): 'intent: card_arrival'


00:30:14 llm: LLM call: 560 prompt tokens, decoding={'do_sample': False}


00:30:15 llm: LLM output (4 new tokens): 'card_delivery_estimate'


00:30:15 llm: LLM call: 534 prompt tokens, decoding={'do_sample': False}


...

00:31:42 baseline: parse_intent: found ['verify_my_identity'] in output -> 'verify_my_identity'


00:31:42 baseline: parse_intent: found ['verify_my_identity'] in output -> 'verify_my_identity'


00:31:42 baseline: parse_intent: found ['verify_my_identity'] in output -> 'verify_my_identity'


00:31:42 baseline: parse_intent: found ['country_support'] in output -> 'country_support'


accuracy (LLM + retrieved examples): 0.84
accuracy (5-NN vote, no LLM):        0.77
no intent found: 3
correct answer present in the 5 examples: 94/100

  'How do I track my card?' -> 'intent: card_arrival' (expected card_arrival)

  'How long should my new card take to arrive?' -> 'card_delivery_estimate' (expected card_arrival)

  'What foreign exchange rate will you use?' -> 'exchange_rate' (expected exchange_rate)


Results:

In [16]:
from src.retrieval import retrieve, build_prompt as retrieval_prompt

r = pd.read_csv("evaluation/results/results_retrieval.csv")
print(f"LLM + retrieved examples: {r.correct.mean():.2f}    5-NN vote only: {r.knn_correct.mean():.2f}    no valid intent: {r.predicted.isna().sum()}")
print(f"LLM right where vote wrong: {(r.correct & ~r.knn_correct).sum()}, LLM wrong where vote right: {(~r.correct & r.knn_correct).sum()}")

print("\nretrieved examples for 'How do I track my card?':")
display(retrieve("How do I track my card?")[["text", "category"]])

few_shot = pd.read_csv("evaluation/results/results_few_shot.csv")
tfidf = pd.read_csv("evaluation/results/results_tfidf_logreg.csv")
pd.DataFrame({"system": ["LLM, 5 random examples (E2)", "5-NN vote, no LLM", "TF-IDF + LogReg", "LLM, 5 retrieved examples"],
              "accuracy": [round(few_shot.correct.mean(), 2), round(r.knn_correct.mean(), 2), round(tfidf.correct.mean(), 2), round(r.correct.mean(), 2)]})


LLM + retrieved examples: 0.84    5-NN vote only: 0.77    no valid intent: 3
LLM right where vote wrong: 12, LLM wrong where vote right: 5

retrieved examples for 'How do I track my card?':


,text,category
6544,how do i track the moeney,verify_source_of_funds
82,How can I track my card's delivery?,card_arrival
26,How do I track the card you sent me?,card_arrival
89,Is there a way to track my card?,card_arrival
95,Can you help me track my card?,card_arrival


,system,accuracy
0,"LLM, 5 random examples (E2)",0.43
1,"5-NN vote, no LLM",0.77
2,TF-IDF + LogReg,0.79
3,"LLM, 5 retrieved examples",0.84


## 8. E6 Evaluation and failure analysis (Req 12)

Criteria: accuracy, valid output, format discipline, reproducibility. The full table of 13 systems, 7 failure types and 11 worked cases are in `docs/FAILURES.md`.
Below: the summary across the four main systems and the rows that every system got wrong.

In [17]:
systems = {"zero_shot": "evaluation/results/results_zero_shot.csv", "few_shot_constraint": "evaluation/results/results_few_shot_constraint.csv",
           "structured": "evaluation/results/results_structured.csv", "retrieval": "evaluation/results/results_retrieval.csv"}
cmp = subset.copy()
for k, f in systems.items():
    cmp[k] = pd.read_csv(f).correct
cmp["n_right"] = cmp[list(systems)].sum(axis=1)
print({k: round(cmp[k].mean(), 2) for k in systems})
print(f"right in all 4: {(cmp.n_right == 4).sum()}, wrong in all 4: {(cmp.n_right == 0).sum()}")
retr = pd.read_csv("evaluation/results/results_retrieval.csv")
cmp.loc[cmp.n_right == 0, ["text", "category"]].assign(retrieval_output=retr.loc[cmp.n_right == 0, "output"])

{'zero_shot': np.float64(0.41), 'few_shot_constraint': np.float64(0.43), 'structured': np.float64(0.43), 'retrieval': np.float64(0.84)}
right in all 4: 24, wrong in all 4: 12


,text,category,retrieval_output
4,Why am I unable to take money from my card but the statement says the transa...,pending_cash_withdrawal,pending_transfer
9,What all currencies can be exchanged?,exchange_via_app,supported_cards_and_currencies
10,How do I exchange currency?,exchange_via_app,intent: supported_cards_and_currencies
29,On my last transaction it seem that my top-up was not successful.,top_up_reverted,top_up_failed
36,Why did I need to verfiy my identity?,why_verify_identity,verify_my_identity
49,Trouble transferring transaction contact for help,declined_transfer,intent not specified
63,"I am attempting to make a purchase online, but my card is not working. I kn...",reverted_card_payment?,transaction_charged_twice
66,What extra charges are there?,transfer_fee_charged,extra_charge_on_statement
67,Is it possible for me to get money out in a different currency?,receiving_money,supported_cards_and_currencies
68,What all is needed to verify the top-up card?,verify_top_up,verify_my_identity


## 9. E7 Final assistant (Req 10)

`main.py` combines the conclusions: retrieval prompt, greedy decoding for the intent, schema validation with the 77-name enum,
two tools with JSON schemas (`classify_intent`, `handle_intent`), and a sampled free-text reply.

The code fixes the tool order. We tested letting the 1.5B model drive the tool loop on 8 cases and it did not succeed once:
it skipped tools, invented an intent for `handle_intent`, or claimed actions it had not taken. With a small model, control belongs in code.

`handle_intent` is a simulation, as the assignment allows.

Action intents (block a card, cancel a transfer, change a PIN, ...) are irreversible, so the assistant asks for confirmation
before it calls `handle_intent`. `handle(message, pending)` returns the reply and the intent that waits for a yes; the caller
keeps that state between turns. Questions and unsupported requests need no confirmation.

In [18]:
from main import handle, TOOLS, INTENT_SCHEMA

print("tool schemas:", [t["function"]["name"] for t in TOOLS])
pending = None
for m in ["I lost my card, please block it",
          "yes",
          "Why is it asking me to verify my identity",
          "My top-up for Google Pay isn't working",
          "asdf qwerty zzz"]:
    print(f"\nCustomer:  {m}")
    reply, pending = handle(m, pending)
    print(f"Assistant: {reply}")

tool schemas: ['classify_intent', 'handle_intent']

Customer:  I lost my card, please block it


19:18:22 llm: LLM call: 527 prompt tokens, decoding={'do_sample': False}


19:18:32 llm: LLM output (6 new tokens): 'lost_or_stolen_card'


19:18:32 baseline: parse_intent: found ['lost_or_stolen_card'] in output -> 'lost_or_stolen_card'


  [tool] classify_intent({'message': 'I lost my card, please block it'}) -> {'intent': 'lost_or_stolen_card'}
Assistant: I understood your request as: lost or stolen card. Shall I go ahead? Reply yes to confirm.

Customer:  yes
  [tool] handle_intent({'intent': 'lost_or_stolen_card'}) -> {'intent': 'lost_or_stolen_card', 'status': 'done', 'result': 'Card blocked. A replacement card is on its way.', 'reference': 'REF-4315'}


...

19:18:42 llm: LLM call: 591 prompt tokens, decoding={'do_sample': False}


Assistant: Your card is compatible with both Apple Pay and Google Pay. You can add your card from the Wallet app on your device.

Customer:  asdf qwerty zzz


19:18:43 llm: LLM output (13 new tokens): 'None of the provided messages match any of the given intents.'


19:18:43 baseline: parse_intent: found nothing in output -> ''


  [tool] classify_intent({'message': 'asdf qwerty zzz'}) -> {'error': "Could not map the message to a known intent (model said 'None of the provided messages match any of the given intents.')."}
Assistant: Sorry, I could not understand your request. Could you say it in a different way?


### Final evaluation on the held-out subset

Every table above uses `eval_subset.csv`, the development set, which also chose the final system. So the retrieval score is a development score.
`heldout_subset.csv` is 100 other test rows that no experiment touched. The final classifier runs on it once.


In [19]:
%run -m evaluation.final_eval

00:31:53 llm: LLM call: 549 prompt tokens, decoding={'do_sample': False}


00:31:54 llm: LLM output (4 new tokens): 'card_arrival'


00:31:54 baseline: parse_intent: found ['card_arrival'] in output -> 'card_arrival'


00:31:54 llm: LLM call: 532 prompt tokens, decoding={'do_sample': False}


00:31:54 llm: LLM output (4 new tokens): 'card_arrival'


...

00:33:43 baseline: parse_intent: found ['apple_pay_or_google_pay'] in output -> 'apple_pay_or_google_pay'


00:33:43 llm: LLM call: 552 prompt tokens, decoding={'do_sample': False}


00:33:44 llm: LLM output (3 new tokens): 'country_support'


00:33:44 baseline: parse_intent: found ['country_support'] in output -> 'country_support'


held-out accuracy (final system): 0.77
no valid intent: 4


In [20]:
heldout = pd.read_csv("evaluation/results/results_heldout.csv")
tfidf = pd.read_csv("evaluation/results/results_tfidf_logreg.csv")
print(f"final system, held-out: {heldout.correct.mean():.2f}    classical reference, development: {tfidf.correct.mean():.2f}")
heldout.loc[~heldout.correct, ["text", "category", "predicted"]].head(5)


final system, held-out: 0.77    classical reference, development: 0.79


,text,category,predicted
11,What options do I have for holding currencies?,fiat_currency_support,supported_cards_and_currencies
15,Why are there limits on auto top-up?,automatic_top_up,disposable_card_limits
20,My card isn't working,card_not_working,virtual_card_not_working
40,I have a payment listed in error.,card_payment_not_recognised,transaction_charged_twice
41,There's a recent charge on my card that I know I didn't make because I've ne...,card_payment_not_recognised,verify_my_identity


### Talk to the assistant

Run this cell by hand. Type a message in the box and press Enter. Press Enter on an empty line to stop.
Note: "Run All" waits at this cell until you stop the loop.

In [ ]:
from main import handle

print("Banking assistant ready. Press Enter on an empty line to exit.")
pending = None
while True:
    message = input("\nCustomer: ").strip()
    if not message:
        break
    reply, pending = handle(message, pending)
    print("Assistant:", reply)

## 10. Summary

Every number below is read from the `results_*.csv` files written by the cells above.


In [21]:
from src.llm import MODEL
from src.prompts import STYLES
from src.decoding import SETTINGS

res = lambda name: pd.read_csv(f"evaluation/results/results_{name}.csv")
acc = lambda name: res(name).correct.mean()

prompts = {s: acc(s) for s in STYLES}
best_prompt = max(prompts, key=prompts.get)
sampling = [acc(f"decoding_{s}") for s in SETTINGS if s != "greedy"]
structured = res("structured")

pd.DataFrame([
    ("Model", MODEL),
    ("E1 baseline", f"{acc('baseline'):.2f}"),
    ("E2 best instruction prompt", f"{prompts[best_prompt]:.2f} ({best_prompt})"),
    ("E3 decoding", f"greedy {acc('decoding_greedy'):.2f}, sampling {min(sampling):.2f} to {max(sampling):.2f}"),
    ("E4 structured output", f"{(~structured.error.str.startswith('not JSON', na=False)).sum()}/100 JSON, {structured.error.isna().sum()}/100 schema-valid"),
    ("Reference: TF-IDF + LogReg", f"{acc('tfidf_logreg'):.2f} on the development set"),
    ("Retrieval few-shot", f"{acc('retrieval'):.2f} on the development set (5-NN vote alone {res('retrieval').knn_correct.mean():.2f})"),
    ("Final system, held-out", f"{acc('heldout'):.2f} on 100 untouched test rows"),
], columns=["Step", "Result"]).style.hide(axis="index")


Step,Result
Model,Qwen/Qwen2.5-1.5B-Instruct
E1 baseline,0.41
E2 best instruction prompt,0.43 (few_shot)
E3 decoding,"greedy 0.43, sampling 0.38 to 0.44"
E4 structured output,"100/100 JSON, 86/100 schema-valid"
Reference: TF-IDF + LogReg,0.79 on the development set
Retrieval few-shot,0.84 on the development set (5-NN vote alone 0.77)
"Final system, held-out",0.77 on 100 untouched test rows


Final system: retrieval + greedy + schema + code-driven tools + sampled reply.

Limits: small model on a single GPU or an 8 GB laptop; `handle_intent` is a stub; TF-IDF retrieval matches words, not meaning (see `docs/FAILURES.md`, type F).
